In [19]:
# 資料集簡單說明
# cluster 0 => split 8
# cluster 1 => split 9 
# 可以直接傳參數call data set

# split 1~5 則是按年齡由小到大排序
# 每個split 包含20%的受試者

# 不管怎樣這三串一定要加

In [ ]:
!pip install torchsummary
!pip install pytorch-metric-learning

In [ ]:
import os
import re
import time
print(os.getcwd())
!cp -r /kaggle/input/drunkgait/colab /kaggle/working/
!mkdir -p /kaggle/working/data
!cp -r /kaggle/input/dataset/drunk71_small /kaggle/working/data
print(f"Now dir : {os.getcwd()}")
# 複製後要記得到working/drunkgait

In [ ]:
import numpy as np
from colab.tools import prinfo2     # log printer
import gc, torch
from colab.SSO import SSO

# 自己設定參數使用教學

In [ ]:
feature = 'j'
model = 'stgcn-raw'

for split in [8,9]:
    print(f"SPLIT : {split}")
    
    for run in range(30):

        command = (
            f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu_f1 "
            f"--batch 128 --data drunk --feature {feature} "
            f"--epochs 80 --split {split} --cls 4 --model {model} --lr 0.1 --loss CE "
        )
        # 使用 `!` 運行 Shell 命令
        b = !{command}
        # print(b)
        
        # 使用 `prinfo()` 解析結果
        try:
            (
                train_cost, train_time, test_time, val_acc_4c, val_acc_3c, val_acc_2c, 
                test_acc_4c, test_acc_3c, test_acc_2c, f1, pre, rec, f1_4c, f1_3c, log
            ) = prinfo2(b)
            
            print(f"Run : {run:>3} | Train_cost : {train_cost:>8.3f}s | Train_time : {train_time:>8.7f}s | "
                      f"Test_time : {test_time:>8.3f}s | Val_acc_4c : {val_acc_4c:>10.7f} | "
                      f"Val_acc_3c : {val_acc_3c:>10.7f} | Val_acc_2c : {val_acc_2c:>10.7f} | "
                      f"Test_acc_4c : {test_acc_4c:>10.7f} | Test_acc_3c : {test_acc_3c:>10.7f} | "
                      f"Test_acc_2c : {test_acc_2c:>10.7f} | F1 : {f1:>10.7f} | "
                      f"F1_4c : {f1_4c:>10.7f} | F1_3c : {f1_3c:>10.7f} | Precision : {pre:>10.7f} | Recall : {rec:>10.7f} | Log : {log}")

        except Exception as e:
            print(f"❌ Error in split {split}, run {run}: {e}")

# 使用SSO找好的參數複現結果

In [ ]:
# SSO-ST-GCN*
# command 要使用 下面這個 這個會有多 --mt，這是把triplet loss打開的選項
# command = f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu_f1 {param_args} --mt"

# Setting = {
#     1 : (4.0, 197.0, 1.0, 1.0, 0.0, 0.0, 0.0, 25.0, 12.0, 74.0, 90.0, 44.0, 50.0),
#     2 : (2.0, 226.0, 1.0, 1.0, 0.0, 0.0, 24.0, 21.0, 77.0, 92.0, 61.0, 37.0, 46.0),
#     3 : (2.0, 204.0, 2.0, 3.0, 2.0, 0.0, 0.0, 22.0, 90.0, 36.0, 59.0, 55.0, 20.0),
#     4 : (2.0, 201.0, 3.0, 1.0, 1.0, 0.0, 22.0, 17.0, 96.0, 83.0, 59.0, 83.0, 52.0),
#     5 : (2.0, 212.0, 1.0, 2.0, 1.0, 0.0, 21.0, 17.0, 22.0, 75.0, 82.0, 99.0, 21.0),
#     8 : (2.0, 227.0, 3.0, 1.0, 3.0, 0.0, 13.0, 26.0, 13.0, 76.0, 90.0, 40.0, 27.0),
#     9 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
#     12 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
#     13 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
#     14 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
#     15 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
# }

###################################################################################################################################

# SSO-STGCN++  ******最後兩位數只是隨便設定的，無實質性作用*********
# command 要使用 下面這個
# command = f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu_f1 {param_args}"

Setting = {
    1 : (3.0, 253.0, 1.0, 1.0, 1.0, 0.0, 16.0, 24.0, 22.0, 11.0, 67.0, 62.0, 21.0),
    2 : (2.0, 248.0, 1.0, 1.0, 3.0, 0.0, 26.0, 28.0, 71.0, 92.0, 74.0, 58.0, 70.0),
    3 : (2.0, 132.0, 1.0, 1.0, 0.0, 0.0, 19.0, 23.0, 70.0, 65.0, 76.0, 46.0, 30.0),
    4 : (3.0, 211.0, 1.0, 1.0, 2.0, 0.0, 17.0, 19.0, 41.0, 81.0, 61.0, 64.0, 50.0),
    5 : (2.0, 189.0, 1.0, 1.0, 1.0, 0.0, 16.0, 30.0, 75.0, 93.0, 57.0, 80.0, 89.0),
    8 : (2.0, 109.0, 1.0, 1.0, 1.0, 0.0, 18.0, 25.0, 85.0, 65.0, 62.0, 80.0, 89.0),
    9 : (2.0, 173.0, 1.0, 1.0, 3.0, 0.0, 29.0, 18.0, 33.0, 38.0, 76.0, 80.0, 89.0),
    12 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
    13 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
    14 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
    15 : (2.0, 214.0, 1.0, 1.0, 3.0, 0.0, 17.0, 27.0, 51.0, 68.0, 58.0, 38.0, 51.0),
}

###################################################################################################################################

for split in [8, 9]:

    print(f"SPLIT : {split}")
    X = Setting[split]

    params_range = {
    'num_init'      : (1, 4),
    'base_channel'  : (64, 256),
    'stride_init'   : (1, 3),
    'tkernel_init'  : (1, 3),
    
    'act'           : (0, 3),           # activate function  
    'opt'           : (0, 2),           # optimizer

    'dropout_bk'    : (15, 30),         # dropout in block *10-2
    'dropout_fc'    : (15, 30),         # dropout in fc *10-2

    'lr'            : (1, 100),         # learning rate *10-3
    'weight_decay'  : (0, 100),         # weight_decay *10-4
    'momentum'      : (50, 99),          # momentum * 10-2
    'margin'        : (20, 100),          # margin in Triplet Loss *10-2
    'lambda_val'    : (0, 90),          # The ratio of Triplet Loss compare to Cross Entropy *10-2
    }

    # 設定初始解
    params = {
        'data'          : 'drunk',          # 我自己讀特定資料集的方式
        'batch'         : 128,              # 可以調整~~
        'epoch'         : 80,               # 可以調整~~
        'feature'       : 'j',              # 我自己讀特定資料集的方式
        'split'         : split,
        
        'num_init'      : 4,
        'base_channel'  : 64,           # output_channel in init_layer
        'stride_init'   : 1,
        'tkernel_init'  : 3,
    
        'num_in'        : 0,                # num_layers_input_stream 固定捨棄後面兩個stream
        'num_main'      : 0,                # num_layers_main_stream 固定捨棄後面兩個stream
    
        'act'           : 0,                # activate function
        'opt'           : 0,                # optimizer
        
        'dropout_bk'    : 0,                # dropout in block
        'dropout_fc'    : 0,                # dropout in fc
    
        'lr'            : 100,              # learning rate
        'weight_decay'  : 5,                # weight_decay
        'momentum'      : 90,               # momentum
        'margin'        : 0.6,              # margin in Triplet Loss
        'lambda_val'    : 0.5,              # The ratio of Triplet Loss compare to Cross Entropy
    }
                                                                                                            
    # 不用動
    def get_param(X, keys=params_range.keys(), params=params):
        """
        將搜尋空間中編碼過的數值 X 映射回實際的模型參數設定，統一保留4位小數。
    
        Parameters:
            X (np.ndarray): 搜尋演算法產出的參數值列表。
            keys (list): 與 X 對應的參數名稱。
            params (dict): 預設參數字典。
    
        Returns:
            dict: 還原為實際值後的參數字典。
        """
        param = params.copy()
    
        for i, key in enumerate(keys):
            val = X[i]
            boundary = params_range[key]
    
            # 統一保留 4 位小數的縮放邏輯
            if key in ['dropout_bk', 'dropout_fc']:
                param[key] = round(val * 1e-2, 4)
            elif key == 'tkernel_init':
                param[key] = int(val * 2 + 1)
            elif key == 'lr':
                param[key] = round(val * 1e-3, 4)
            elif key == 'weight_decay':
                param[key] = round(val * 1e-4, 4)
            elif key == 'momentum':
                param[key] = round(val * 1e-2, 4)
            elif key == 'margin':
                param[key] = round(val * 1e-2, 4)
            elif key == 'lambda_val':
                param[key] = round(val * 1e-2, 4)
            elif (
                isinstance(boundary, (tuple, list)) and
                all(isinstance(b, int) for b in boundary)
            ):
                param[key] = int(val)
            else:
                param[key] = val
    
        return param
    
    def fitness(X, run=None):
        # **釋放 GPU 記憶體**
        torch.cuda.empty_cache()
        
        param = get_param(X=X)
        param_args = " ".join([f"--{key} {str(value).strip()}" for key, value in param.items()])

        # for SSO-STGCN++
        command = f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu_f1 {param_args}"
        
        # for SSO-ST-GCN*
        # command = f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu_f1 {param_args} --mt"

        try:
            # 使用 `!{command}` 在 Jupyter Notebook / Colab 內執行
            output = !{command}
            # print(output)
            # 解析輸出
            train_cost, train_time, test_time, val_acc_4c, val_acc_3c, val_acc_2c, \
            test_acc_4c, test_acc_3c, test_acc_2c, f1, pre, rec, f1_4c, f1_3c, log = prinfo2(output)
            
            if run is not None:
                print(f"Run : {run:>3} | Train_cost : {train_cost:>8.3f}s | Train_time : {train_time:>8.7f}s | "
                      f"Test_time : {test_time:>8.3f}s | Val_acc_4c : {val_acc_4c:>10.7f} | "
                      f"Val_acc_3c : {val_acc_3c:>10.7f} | Val_acc_2c : {val_acc_2c:>10.7f} | "
                      f"Test_acc_4c : {test_acc_4c:>10.7f} | Test_acc_3c : {test_acc_3c:>10.7f} | "
                      f"Test_acc_2c : {test_acc_2c:>10.7f} | F1 : {f1:>10.7f} | "
                      f"F1_4c : {f1_4c:>10.7f} | F1_3c : {f1_3c:>10.7f} | Precision : {pre:>10.7f} | Recall : {rec:>10.7f} | Log : {log}")
                
        except Exception as e:
            print(f"❌ 發生錯誤: {e}")
            # 確保所有變數有值，避免 `NoneType` 出錯
            train_cost = train_time = test_time = 0
            val_acc_4c = val_acc_3c = val_acc_2c = 0
            test_acc_4c = test_acc_3c = test_acc_2c = 0
            f1_4c = f1_3c = f1 = pre = rec = 0
            log = ""
    
        # 確保 `NoneType` 不影響計算
        fitness_value = (val_acc_4c or 0)
    
        # 確保 `log` 不為 `None`
        log = log if log is not None else ""
    
        record_message = {
            'train_cost': train_cost,
            'train_time': train_time,
            'test_time': test_time,
            'val_acc_4c': val_acc_4c,
            'val_acc_3c': val_acc_3c,
            'val_acc_2c': val_acc_2c,
            'test_acc_4c': test_acc_4c,
            'test_acc_3c': test_acc_3c,
            'test_acc_2c': test_acc_2c,
            'f1': f1,
            'precision': pre,
            'recall': rec,
            'f1_4c': f1_4c,
            'f1_3c': f1_3c,
            'log': log
        }
        return fitness_value, record_message
    
    for run in range(30):
        fitness_value, record_message = fitness(X=X, run=run)

# SSO RUN

In [ ]:
split = 8 # 資料集編號
run = 7 # 提供方便記錄的編號，代表你跑第幾次，可以讓此變數作為for loop內的iter.

In [ ]:
params_range = {
    'num_init'      : (1, 4),
    'base_channel'  : (64, 256),
    'stride_init'   : (1, 3),
    'tkernel_init'  : (1, 3),
    
    'act'           : (0, 3),           # activate function  
    'opt'           : (0, 2),           # optimizer

    'dropout_bk'    : (15, 30),         # dropout in block *10-2
    'dropout_fc'    : (15, 30),         # dropout in fc *10-2

    'lr'            : (1, 100),         # learning rate *10-3
    'weight_decay'  : (0, 100),         # weight_decay *10-4
    'momentum'      : (50, 99),          # momentum * 10-2
    'margin'        : (20, 100),          # margin in Triplet Loss *10-2
    'lambda_val'    : (0, 90),          # The ratio of Triplet Loss compare to Cross Entropy *10-2
}

# 設定初始解
params = {
    'data'          : 'drunk',          # 我自己讀特定資料集的方式
    'batch'         : 128,              # 可以調整~~
    'epoch'         : 80,               # 可以調整~~
    'feature'       : 'j',              # 我自己讀特定資料集的方式
    'split'         : split,            # 接上面設定的資料集編號
    
    'num_init'      : 4,
    'base_channel'  : 64,           # output_channel in init_layer
    'stride_init'   : 1,
    'tkernel_init'  : 1,

    'num_in'        : 0,                # num_layers_input_stream 固定捨棄後面兩個stream
    'num_main'      : 0,                # num_layers_main_stream 固定捨棄後面兩個stream

    'act'           : 0,                # activate function
    'opt'           : 0,                # optimizer
    
    'dropout_bk'    : 0,                # dropout in block
    'dropout_fc'    : 0,                # dropout in fc

    'lr'            : 100,              # learning rate
    'weight_decay'  : 5,                # weight_decay
    'momentum'      : 90,               # momentum
    'margin'        : 60,              # margin in Triplet Loss
    'lambda_val'    : 50,              # The ratio of Triplet Loss compare to Cross Entropy
}
                                                                                                        
# 不用動
def get_param(X, keys=params_range.keys(), params=params):
    """
    將搜尋空間中編碼過的數值 X 映射回實際的模型參數設定，統一保留4位小數。

    Parameters:
        X (np.ndarray): 搜尋演算法產出的參數值列表。
        keys (list): 與 X 對應的參數名稱。
        params (dict): 預設參數字典。

    Returns:
        dict: 還原為實際值後的參數字典。
    """
    param = params.copy()

    for i, key in enumerate(keys):
        val = X[i]
        boundary = params_range[key]

        # 統一保留 4 位小數的縮放邏輯
        if key in ['dropout_bk', 'dropout_fc']:
            param[key] = round(val * 1e-2, 4)
        elif key == 'tkernel_init':
            param[key] = int(val * 2 + 1)
        elif key == 'lr':
            param[key] = round(val * 1e-3, 4)
        elif key == 'weight_decay':
            param[key] = round(val * 1e-4, 4)
        elif key == 'momentum':
            param[key] = round(val * 1e-2, 4)
        elif key == 'margin':
            param[key] = round(val * 1e-2, 4)
        elif key == 'lambda_val':
            param[key] = round(val * 1e-2, 4)
        elif (
            isinstance(boundary, (tuple, list)) and
            all(isinstance(b, int) for b in boundary)
        ):
            param[key] = int(val)
        else:
            param[key] = val

    return param

def fitness(X):
    # **釋放 GPU 記憶體**
    torch.cuda.empty_cache()
    
    param = get_param(X=X)
    param_args = " ".join([f"--{key} {str(value).strip()}" for key, value in param.items()])
    command = f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu_f1 --mt {param_args}"
    
    try:
        # 使用 `!{command}` 在 Jupyter Notebook / Colab 內執行
        output = !{command}
        # print(output)
        # 解析輸出
        train_cost, train_time, test_time, val_acc_4c, val_acc_3c, val_acc_2c, \
            test_acc_4c, test_acc_3c, test_acc_2c, f1, pre, rec, f1_4c, f1_3c, log = prinfo2(output)
        
    except Exception as e:
        print(f"❌ 發生錯誤: {e}")
        # 確保所有變數有值，避免 `NoneType` 出錯
        train_cost = train_time = test_time = 0
        val_acc_4c = val_acc_3c = val_acc_2c = 0
        test_acc_4c = test_acc_3c = test_acc_2c = 0
        f1 = f1_4c = f1_3c = pre = rec = 0
        log = ""

    # 確保 `NoneType` 不影響計算
    fitness_value = (val_acc_4c or 0)

    # 確保 `log` 不為 `None`
    log = log if log is not None else ""

    record_message = {
        'train_cost': train_cost,
        'train_time': train_time,
        'test_time': test_time,
        'val_acc_4c': val_acc_4c,
        'val_acc_3c': val_acc_3c,
        'val_acc_2c': val_acc_2c,
        'test_acc_4c': test_acc_4c,
        'test_acc_3c': test_acc_3c,
        'test_acc_2c': test_acc_2c,
        'f1': f1,
        'f1_4c' : f1_4c,
        'f1_3c' : f1_3c,
        'precision': pre,
        'recall': rec,
        'log': log
    }
    return fitness_value, record_message

# 存檔名稱
save_name = f"{params['data']}_{params['split']}_{348}_run{run-1}_Triplet"

# 把所有東西丟進去
SSO_searcher = SSO(
Ngen=10,                                # 世代數
Nsol=10,                                # 解的數量
Cg=0.3,          # 全局最佳解的權重
Cp=0.4,          # 個體最佳解的權重
Cw=0.8,          # 隨機解的權重
save_name=save_name,                    # 記錄檔儲存前綴
fitness=fitness,                        # 適應度函數
base_param=params,                      # 基礎參數 (可選)
boundary=params_range,                  # 支持 tuple 或 list (詳細說明請參照前述)
direction="maximize",                   # 優化方向 ('maximize', 'minimize')
)

# 運行
SSO_searcher.run()

In [ ]:
'''
# 當然如果有設備也可以

# 存檔名稱
save_name = f"{params['data']}_{params['split']}_{348}_run{run-1}_Triplet"

# 運行
for run in range(30):
    SSO_searcher = SSO(
        Ngen=10,                                # 世代數
        Nsol=10,                                # 解的數量
        Cg=0.3,          # 全局最佳解的權重
        Cp=0.4,          # 個體最佳解的權重
        Cw=0.8,          # 隨機解的權重
        save_name=save_name,                    # 記錄檔儲存前綴
        fitness=fitness,                        # 適應度函數
        base_param=params,                      # 基礎參數 (可選)
        boundary=params_range,                  # 支持 tuple 或 list (詳細說明請參照前述)
        direction="maximize",                   # 優化方向 ('maximize', 'minimize')
        )
    SSO_searcher.run()
'''


# 程式參數會用到的

In [2]:
# str
model = 'stgcn'     # stgcn++
'stgcn-raw'         # stgcn
'dgstcn'            # dgstgcn
'sgn'               # sgn
'aagcn'             # aagcn

# str
feature = 'j'       # joint
'b'                 # bone
'jm'                # joint motion
'bm'                # bone motion


SyntaxError: invalid decimal literal (3704603620.py, line 15)

# 總資料集(未分割/未前處理)

In [16]:
import pickle

file = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\tools\data\drunk\sagittal_yolo_subjects.pkl"

with open(file, 'rb') as f:
    data = pickle.load(f)

print(data.keys())

# 所有受試者編號
print(data['split'].keys())

# 編號43受試者的所有影片
print(data['split']['43'])

# 編號43受試者的所有影片
print(data['split']['43'][0])

# 全部受試者的原始資料
# print(data['annotations'])



# 如果要調用受試者的影片，要找frame_dir跟他影片影稱一樣的
frame_name = '43_sagittal_normal_01'
result = [ann for ann in data['annotations'] if ann['frame_dir'] == frame_name]

if result:
    print(f"✅ 找到影片：{result[0]}")
else:
    print("❌ 找不到該影片")

# 其中 
# 'frame_dir' : 影片名稱
# 'label' : 標籤
# 'keypoint' : 關節點位置(pixel)
# 'bbox' : 人物框位置(pixel)

dict_keys(['split', 'annotations'])
dict_keys(['43', '40', '09', '42', '25', '37', '52', '64', '07', '18', '60', '63', '06', '26', '45', '46', '55', '57', '50', '59', '14', '69', '13', '01', '28', '58', '02', '53', '47', '71', '05', '08', '27', '68', '03', '11', '12', '15', '04', '10', '16', '19', '23', '22', '24', '35', '39', '44', '49', '65', '41', '38', '30', '21', '67', '56', '62', '34', '51', '20', '54', '17', '66', '31', '33', '70', '61', '36', '48', '32', '29'])
['43_sagittal_normal_01', '43_sagittal_normal_02', '43_sagittal_normal_03', '43_sagittal_normal_04', '43_sagittal_normal_05', '43_sagittal_normal_06', '43_sagittal_normal_07', '43_sagittal_normal_08', '43_sagittal_normal_09', '43_sagittal_normal_10', '43_sagittal_low_01', '43_sagittal_low_02', '43_sagittal_low_03', '43_sagittal_low_04', '43_sagittal_low_05', '43_sagittal_low_06', '43_sagittal_low_07', '43_sagittal_low_08', '43_sagittal_low_09', '43_sagittal_low_10', '43_sagittal_medium_01', '43_sagittal_medium_02', '43_s